# CURE-Rec — next actions and paper-readiness notebook

This notebook is the execution checklist after the master run. It separates **cheap postprocessing and validation actions** from **expensive model-search and multi-seed actions**.

## Required order

1. verify the completed Torch BPR run and audit;
2. run the staged validation-only BPR search;
3. inspect the selected final BPR/hybrid configuration;
4. regenerate aggregate assets from completed expensive CURE sweeps without rerunning them;
5. regenerate the controlled oracle regime suite with separate estimated/oracle recovery metrics;
6. archive the reproducibility snapshot;
7. only then plan calibration robustness and SASRec.


## 1. Setup and source verification

Run this cell after `git pull` and a kernel restart. It deliberately clears stale `cure_rec` modules from long-lived VS Code kernels.

In [ ]:
from pathlib import Path
import hashlib
import importlib
import inspect
import json
import shutil
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the CURE-Rec code directory or repository root.')
sys.path[:] = [str(ROOT), *[item for item in sys.path if item != str(ROOT)]]
for name in list(sys.modules):
    if name == 'cure_rec' or name.startswith('cure_rec.'):
        del sys.modules[name]
importlib.invalidate_caches()

from cure_rec.analysis import analyze_dataset
from cure_rec.config import load_settings
from cure_rec.data import load_dataset
from cure_rec.experiments import postprocess_seed_sweep
from cure_rec.regimes import run_regime_suite
from cure_rec.observability import RunLogger
from cure_rec.search import SearchConfig, run_staged_bpr_search
from cure_rec.models import chronological_leave_one_out

assert 'bpr_epochs' in inspect.signature(analyze_dataset).parameters
print('CURE-Rec source:', ROOT)
print('Analysis signature:', inspect.signature(analyze_dataset))


## 2. Configure paths to the completed results

The default paths point to your currently committed runs. Change them only when you intentionally want to inspect a different run.

In [ ]:
RUN_ROOT = ROOT / 'runs'
MOVIELENS_SOURCE = ROOT / 'data' / 'raw' / 'movielens_1m'
TORCH_RUN = RUN_ROOT / 'data-analysis-movielens_1m-20260805T102506Z'
MASTER_RUN = RUN_ROOT / 'all-variations-20260804T202725Z'
FULL_FIVE_SWEEP = MASTER_RUN / 'full_five_seed' / 'seed-sweep-20260804T210624Z'
FULL_TWENTY_SWEEP = MASTER_RUN / 'full_twenty_seed' / 'seed-sweep-20260805T000401Z'
QUICK_CONFIG = ROOT / 'configs' / 'curesim_quickstart.yaml'
FULL_CONFIG = ROOT / 'configs' / 'curesim_full.yaml'

for path in [TORCH_RUN, MASTER_RUN, FULL_FIVE_SWEEP, FULL_TWENTY_SWEEP]:
    print(('FOUND' if path.exists() else 'MISSING'), path)


## 3. Action 1 — inspect the completed Torch BPR run

This is cheap and should be done before any new search. The critical audit violations must all be zero. Candidate coverage must be shared across popularity, Torch BPR, and hybrid.

In [ ]:
if not TORCH_RUN.exists():
    raise FileNotFoundError(f'Completed Torch run not found: {TORCH_RUN}')

metrics = pd.read_csv(TORCH_RUN / 'tables' / 'data_table_model_metrics.csv')
eval_audit = pd.read_csv(TORCH_RUN / 'tables' / 'data_table_evaluation_audit.csv')
pairwise = pd.read_csv(TORCH_RUN / 'tables' / 'data_table_pairwise_accuracy.csv')
hybrid_validation = pd.read_csv(TORCH_RUN / 'tables' / 'data_table_hybrid_validation.csv')
manifest = json.loads((TORCH_RUN / 'artifacts' / 'analysis_manifest.json').read_text())

display(metrics)
display(eval_audit)
display(pairwise)
display(hybrid_validation.sort_values('ndcg_at_k', ascending=False))
print('Manifest backend:', manifest.get('bpr_backend'))
print('Selected hybrid alpha:', manifest.get('selected_hybrid_alpha'))


## 4. Action 2 — enforce the evaluation audit gate

Do not proceed to search or SASRec if this cell fails. The assertions establish that metric differences are not produced by candidate leakage, missing targets, or ranking-direction errors.

In [ ]:
critical = [
    'seen_item_violations',
    'missing_target_violations',
    'candidate_equality_violations',
    'descending_score_violations',
]
for column in critical:
    assert (eval_audit[column] == 0).all(), f'Audit failure in {column}'

for _, row in eval_audit.dropna(subset=['best_validation_epoch']).iterrows():
    assert row['restored_checkpoint_epoch'] == row['best_validation_epoch'], 'Best checkpoint was not restored'

print('Evaluation audit passed.')
print('Candidate coverage:', metrics[['model', 'candidate_coverage', 'cold_test_items']].to_dict(orient='records'))
print('Pairwise diagnostics:', pairwise.to_dict(orient='records'))


## 5. Action 3 — staged Torch BPR search

Enable this only after the audit gate passes. Stage A searches optimization and negative strategy; Stage B searches capacity and batch size; Stage C selects hybrid alpha using validation only. Test metrics are evaluated once after configuration selection.

In [ ]:
RUN_STAGED_BPR_SEARCH = False
SEARCH_OUTPUT = RUN_ROOT / 'bpr-search-movielens-final'

if RUN_STAGED_BPR_SEARCH:
    ml1m = load_dataset('movielens_1m', MOVIELENS_SOURCE, download=True)
    split = chronological_leave_one_out(ml1m.interactions)
    final_search = run_staged_bpr_search(
        split,
        SEARCH_OUTPUT,
        SearchConfig(stage_epochs=40, final_epochs=200, max_eval_users=1_000, top_k_stage_a=3, seed=42),
    )
    print('Search output:', SEARCH_OUTPUT)
    display(final_search)
else:
    print('Staged BPR search disabled. Set RUN_STAGED_BPR_SEARCH = True when ready.')


## 6. Action 4 — inspect staged search output

Run this after Stage 3 completes. The final result should be selected by validation NDCG@10, not by test metrics.

In [ ]:
if SEARCH_OUTPUT.exists() and (SEARCH_OUTPUT / 'bpr_search_final_test.csv').exists():
    stage_a = pd.read_csv(SEARCH_OUTPUT / 'bpr_search_stage_a.csv')
    stage_b = pd.read_csv(SEARCH_OUTPUT / 'bpr_search_stage_b.csv')
    stage_c = pd.read_csv(SEARCH_OUTPUT / 'bpr_search_stage_c.csv')
    final_test = pd.read_csv(SEARCH_OUTPUT / 'bpr_search_final_test.csv')
    search_manifest = json.loads((SEARCH_OUTPUT / 'bpr_search_manifest.json').read_text())
    display(stage_a.head(10))
    display(stage_b.head(10))
    display(stage_c)
    display(final_test)
    print('Selected config:', search_manifest)
else:
    print('No completed staged search found yet.')


## 7. Action 5 — regenerate aggregate assets from the completed full sweeps

This is cheap. It reuses the completed raw coalition artifacts and does not repeat the expensive 5-seed or 20-seed CURE-Sim evaluations.

In [ ]:
RUN_POSTPROCESS_SWEEPS = True

if RUN_POSTPROCESS_SWEEPS:
    full_settings = load_settings(FULL_CONFIG)
    for sweep in [FULL_FIVE_SWEEP, FULL_TWENTY_SWEEP]:
        if not sweep.exists():
            print('Missing sweep:', sweep)
            continue
        rebuilt = postprocess_seed_sweep(sweep, full_settings)
        print('Postprocessed:', rebuilt.run_dir)
        display(rebuilt.decisions)
        display(rebuilt.base_feasibility)
else:
    print('Postprocessing disabled.')


## 8. Action 6 — regenerate controlled oracle regime metrics

This is cheap. It produces the corrected separation between estimated-game recovery and oracle-game recovery, including oracle regret in the misspecified regime.

In [ ]:
RUN_REGIME_REFRESH = True

if RUN_REGIME_REFRESH:
    regime_settings = load_settings(QUICK_CONFIG)
    regime_settings.run.name = 'curesim-regime-refresh'
    regime_settings.run.output_root = RUN_ROOT
    regime_logger = RunLogger(regime_settings)
    try:
        regime_refresh = run_regime_suite(regime_settings, regime_logger)
        regime_logger.close(status='completed')
    except Exception:
        regime_logger.close(status='failed')
        raise
    print('Regime refresh:', regime_refresh.run_dir)
    display(regime_refresh.summary[[
        'regime', 'expected_estimated_selected', 'oracle_selected',
        'observed_estimated_selected', 'estimated_selection_match',
        'oracle_selection_match', 'oracle_regret',
    ]])
    display(regime_refresh.attribution_recovery.groupby('regime', as_index=False).agg(
        shapley_mae=('absolute_error', 'mean'),
        sign_accuracy=('sign_correct', 'mean'),
        point_coverage=('covered_by_estimated_point_region', 'mean'),
    ))
else:
    print('Regime refresh disabled.')


## 9. Action 7 — archive the reproducibility snapshot

This cell copies the small metadata, summary tables, and hashes into an archive folder. It intentionally does not duplicate raw MovieLens data or the entire large result tree. Use the archive as the local precursor to a Zenodo/OSF/release snapshot.

In [ ]:
ARCHIVE_RESULTS = False
ARCHIVE_DIR = ROOT.parent / "results" / "reproducibility_snapshot_latest"

if ARCHIVE_RESULTS:
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    include = [
        TORCH_RUN / "artifacts" / "analysis_manifest.json",
        TORCH_RUN / "tables" / "data_table_model_metrics.csv",
        TORCH_RUN / "tables" / "data_table_evaluation_audit.csv",
        TORCH_RUN / "tables" / "data_table_pairwise_accuracy.csv",
        MASTER_RUN / "all_variations_summary.csv",
        MASTER_RUN / "all_variations_seed_decisions.csv",
    ]
    checksums = []
    for source in include:
        if not source.exists():
            print("Skipping missing:", source)
            continue
        target = ARCHIVE_DIR / source.name
        shutil.copy2(source, target)
        checksums.append(f"{hashlib.sha256(target.read_bytes()).hexdigest()}  {target.name}")

    (ARCHIVE_DIR / "SHA256SUMS.txt").write_text("\n".join(checksums) + "\n")
    (ARCHIVE_DIR / "REPRODUCE.md").write_text(
        "Source branch: arena/019fcbf7-next-paper\n"
        "Run external data analysis, regimes, then postprocess seed sweeps from this notebook.\n"
    )
    print("Archive written:", ARCHIVE_DIR)
else:
    print("Archive disabled. Enable after selecting final external and CURE-Sim results.")


## 10. Remaining action — calibration robustness and SASRec

Do not implement or run these until the BPR search and audit are accepted. The next development task is a calibration sweep over fatigue strength, repetition threshold, horizon, provider threshold, provider-balancing strength, novelty benefit, and exploration cost. Then add SASRec using the same shared warm-item candidate evaluator and audit output.

At that point, rerun only the selected final configurations across multiple seeds; do not rerun the whole master plan blindly.